# Fine-tune Qwen3.5-4B → MITRE ATT&CK Specialist (bf16 GPU)

**Requires a bf16 GPU.** T4 / P100 (Kaggle free, Colab free) **cannot** train Qwen3.5 — its GatedDeltaNet is bf16-native (fp16 is numerically broken, fp32 OOMs on 16 GB). Use one of:
- **Colab Pro → Runtime → Change runtime type → L4 or A100**
- a rented **L4 / A10 / A100** (RunPod / Vast / Lambda, ~$0.4–$0.6/hr, this job <$1)

16-bit LoRA, 1 epoch (from `ft_config.py`). Flow: install → bf16 check → clone → upload dataset → smoke test → train + GGUF → download.
Then on your machine: `ollama create mitre-qwen3.5:4b -f export/Modelfile.qwen35`.

In [ ]:
# 1. Install Unsloth + unsloth_zoo (pulls transformers v5, REQUIRED for Qwen3.5).
!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
# The protobuf bump above breaks the preinstalled wandb (cannot import 'Imports'
# from wandb_telemetry_pb2). We don't use wandb (report_to="none"), so remove it.
!pip uninstall -y -q wandb
import os; os.environ["WANDB_DISABLED"] = "true"
import transformers
print("transformers", transformers.__version__, "(need >= 5.0)")
assert int(transformers.__version__.split('.')[0]) >= 5, \n    "transformers < 5 — restart the runtime (Runtime → Restart) and re-run this cell."

In [ ]:
# 2. GPU check — Qwen3.5 NEEDS bf16. Fail fast on a T4/P100 instead of crashing
#    mid-training with 'BFloat16 != Half' in the GatedDeltaNet.
import torch
name = torch.cuda.get_device_name(0)
print("device:", name, "| bf16:", torch.cuda.is_bf16_supported())
assert torch.cuda.is_bf16_supported(), (
    f"{name} has no bf16. Qwen3.5's GatedDeltaNet needs bf16 (fp16 breaks, fp32 "
    "OOMs on 16GB). Switch to an L4 / A10 / A100 runtime.")
print("bf16 OK -> good to train")

In [ ]:
# 3. Clone the repo (train_unsloth.py + ft_config.py + templates) and cd in.
#    Push your finetune changes to this branch first, or the clone is stale.
BRANCH = "main"
REPO = "https://github.com/NitithX374/CyberCase-Intelligence-Framework.git"
!rm -rf CyberCase-Intelligence-Framework
!git clone --depth 1 -b $BRANCH $REPO
%cd CyberCase-Intelligence-Framework/rag_service/finetune

In [ ]:
# 4. Provide the locally-built dataset (data/output is gitignored).
#    Build first on your machine:  python data/build_dataset.py --max-per-category 600
import os
os.makedirs("data/output", exist_ok=True)
try:
    from google.colab import files            # Colab: upload the two files
    print("Select train.jsonl and val.jsonl ...")
    up = files.upload()
    for fn in up:
        os.replace(fn, f"data/output/{fn}")
except ImportError:
    print("Not on Colab — scp train.jsonl & val.jsonl into data/output/ yourself.")
!wc -l data/output/*.jsonl

In [ ]:
# 5. Smoke test (5 steps) — confirm the loop runs before the full job.
!python train/train_unsloth.py --max-steps 5

In [ ]:
# 6. Full training (1 epoch, set in ft_config.py) + GGUF export (Q4_K_M).
#    The adapter is saved BEFORE the GGUF step, so it survives a GGUF failure.
!python train/train_unsloth.py --gguf

In [ ]:
# 7. Download the GGUF (and always back up the LoRA adapter).
import glob, os, shutil
ggufs = glob.glob("export/outputs/gguf/*[Qq]4_[Kk]_[Mm]*.gguf")
adapter = "train/outputs/mitre-qwen-lora"
if os.path.isdir(adapter):
    shutil.make_archive("mitre-qwen3.5-lora", "zip", adapter)
    print("adapter zip: mitre-qwen3.5-lora.zip")
try:
    from google.colab import files
    if ggufs:
        print("GGUF:", ggufs[0]); files.download(ggufs[0])
    elif os.path.exists("mitre-qwen3.5-lora.zip"):
        print("No GGUF — downloading adapter zip to convert later.")
        files.download("mitre-qwen3.5-lora.zip")
except ImportError:
    print("GGUF at:", ggufs[0] if ggufs else "(none — use the adapter zip)")

## On your machine (keeps stock `qwen3.5:4b` intact)
Download `mitre-qwen3.5-4b-Q4_K_M.gguf`, then:
```powershell
cd rag_service/finetune
# put the .gguf in export/outputs/gguf/ (Modelfile.qwen35's FROM already points there)
ollama create mitre-qwen3.5:4b -f export/Modelfile.qwen35
ollama run mitre-qwen3.5:4b "What is T1059?"   # direct answer, no <think>
```
### A/B compare
```powershell
python compare/run_comparison.py --max-samples 20   # qwen3.5:4b vs mitre-qwen3.5:4b
```
Needs Neo4j + Qdrant up & ingested, and Ollama serving both models.